In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json, cv2, math
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

ROOT = Path('/content/drive/MyDrive/TrafficAI_ControlCenter')
RAW = ROOT / 'datasets' / 'raw' / 'aicity2021_t4'
PROC = ROOT / 'datasets' / 'processed' / 'module2_breakdown'
PROC.mkdir(parents=True, exist_ok=True)

events = pd.read_parquet(PROC / 'anomaly_events.parquet')
print(events.head())
print('events', len(events), 'matched', events['video_path'].notna().sum())

Mounted at /content/drive
   video_id  start_sec  end_sec  duration_sec  \
0         2      587.0    894.0         307.0   
1         9        0.0    287.0         287.0   
2        11        0.0    888.0         888.0   
3        14      475.0    600.0         125.0   
4        33        0.0     90.0          90.0   

                                          video_path  label  
0  /content/drive/MyDrive/TrafficAI_ControlCenter...      1  
1  /content/drive/MyDrive/TrafficAI_ControlCenter...      1  
2  /content/drive/MyDrive/TrafficAI_ControlCenter...      1  
3  /content/drive/MyDrive/TrafficAI_ControlCenter...      1  
4  /content/drive/MyDrive/TrafficAI_ControlCenter...      1  
events 18 matched 18


In [2]:
# Find train video folder
vids = sorted(RAW.rglob('*.mp4'))
vid_map = {int(p.stem): p for p in vids if p.stem.isdigit()}
print('videos indexed', len(vid_map))

# anomaly dict: video_id -> list[(start,end)]
anom = {}
for _, r in events.iterrows():
    anom.setdefault(int(r['video_id']), []).append((float(r['start_sec']), float(r['end_sec'])))

def is_anomaly(vid, t):
    for s,e in anom.get(vid, []):
        if s <= t <= e:
            return 1
    return 0

videos indexed 150


In [3]:
STRIDE = 5.0          # seconds between windows
MAX_VIDEOS = 80     # set e.g. 80 for a quicker first run; None = all
FRAME_COMPARE_GAP = 10  # frames apart for motion

rows = []
video_ids = sorted(vid_map.keys())
if MAX_VIDEOS:
    video_ids = video_ids[:MAX_VIDEOS]

for vid in tqdm(video_ids, desc='Module2 videos'):
    path = str(vid_map[vid])
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        continue
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    nframes = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    dur = nframes / fps if fps > 0 else 0
    if dur < 10:
        cap.release()
        continue

    times = np.arange(0, max(dur - 1, 1), STRIDE)
    for t in times:
        f0 = int(t * fps)
        f1 = min(f0 + FRAME_COMPARE_GAP, nframes - 1)

        cap.set(cv2.CAP_PROP_POS_FRAMES, f0)
        ok0, fr0 = cap.read()
        cap.set(cv2.CAP_PROP_POS_FRAMES, f1)
        ok1, fr1 = cap.read()
        if not (ok0 and ok1):
            continue

        g0 = cv2.cvtColor(fr0, cv2.COLOR_BGR2GRAY)
        g1 = cv2.cvtColor(fr1, cv2.COLOR_BGR2GRAY)
        g0 = cv2.resize(g0, (320, 180))
        g1 = cv2.resize(g1, (320, 180))

        # frame difference energy
        diff = cv2.absdiff(g0, g1)
        motion_mean = float(diff.mean())
        motion_std = float(diff.std())
        motion_p95 = float(np.percentile(diff, 95))

        # “stationary-ish” if motion is very low
        stationary_score = float(1.0 / (1.0 + motion_mean))

        # local context: motion relative to video median later; store raw now
        label = is_anomaly(vid, float(t))

        rows.append({
            'video_id': vid,
            't_sec': float(t),
            'duration_sec': float(dur),
            'fps': float(fps),
            'motion_mean': motion_mean,
            'motion_std': motion_std,
            'motion_p95': motion_p95,
            'stationary_score': stationary_score,
            'rel_time': float(t / dur),
            'has_any_anomaly_in_video': int(vid in anom),
            'y': int(label),  # 1=incident window, 0=normal
            'video_path': path,
        })
    cap.release()

m2 = pd.DataFrame(rows)
print('windows', len(m2), 'positives', int(m2['y'].sum()), 'negatives', int((m2['y']==0).sum()))
print(m2.head())

Module2 videos:   0%|          | 0/80 [00:00<?, ?it/s]

windows 14110 positives 1272 negatives 12838
   video_id  t_sec  duration_sec   fps  motion_mean  motion_std  motion_p95  \
0         1    0.0    894.033333  30.0     5.908299   13.766434        26.0   
1         1    5.0    894.033333  30.0     7.140920   14.214355        32.0   
2         1   10.0    894.033333  30.0     7.297986   14.679794        34.0   
3         1   15.0    894.033333  30.0     7.577795   16.597973        37.0   
4         1   20.0    894.033333  30.0     6.397812   14.663673        30.0   

   stationary_score  rel_time  has_any_anomaly_in_video  y  \
0          0.144753  0.000000                         0  0   
1          0.122836  0.005593                         0  0   
2          0.120511  0.011185                         0  0   
3          0.116580  0.016778                         0  0   
4          0.135175  0.022371                         0  0   

                                          video_path  
0  /content/drive/MyDrive/TrafficAI_ControlCenter...

In [4]:
# normalize motion within each video (better than global thresholds)
m2['motion_mean_z'] = m2.groupby('video_id')['motion_mean'].transform(
    lambda s: (s - s.mean()) / (s.std() + 1e-6)
)
m2['low_motion_flag'] = (m2['motion_mean_z'] < -0.5).astype(int)

# optional rule-assisted soft score (not final label; label stays AI City ground truth)
m2['rule_stop_hint'] = (
    (m2['stationary_score'] > m2.groupby('video_id')['stationary_score'].transform('median'))
    & (m2['low_motion_flag'] == 1)
).astype(int)

print(m2.groupby('y')[['motion_mean','stationary_score','motion_mean_z']].mean())

feat_cols = [
    't_sec','duration_sec','rel_time','fps',
    'motion_mean','motion_std','motion_p95','stationary_score',
    'motion_mean_z','low_motion_flag','rule_stop_hint','has_any_anomaly_in_video'
]

ml = m2[feat_cols + ['y','video_id']].copy()
ml.to_parquet(PROC / 'module2_ml_table.parquet', index=False)
ml.to_csv(PROC / 'module2_ml_table_sample.csv', index=False)

meta = {
    'rows': int(len(ml)),
    'positives': int(ml['y'].sum()),
    'negatives': int((ml['y']==0).sum()),
    'features': feat_cols,
    'label_source': 'AI City 2021 Track4 train-anomaly-results.txt',
    'feature_source': 'OpenCV frame-difference motion features on real CCTV clips',
    'note': 'y=1 if window time falls inside annotated anomaly interval'
}
(PROC / 'module2_ml_meta.json').write_text(json.dumps(meta, indent=2))
print(json.dumps(meta, indent=2))

ok = len(ml) > 500 and ml['y'].sum() > 0 and (ml['y']==0).sum() > 0
print('STEP 6 OK' if ok else 'STEP 6 INCOMPLETE')

   motion_mean  stationary_score  motion_mean_z
y                                              
0     3.345745          0.352154      -0.004088
1     5.453048          0.228143       0.041264
{
  "rows": 14110,
  "positives": 1272,
  "negatives": 12838,
  "features": [
    "t_sec",
    "duration_sec",
    "rel_time",
    "fps",
    "motion_mean",
    "motion_std",
    "motion_p95",
    "stationary_score",
    "motion_mean_z",
    "low_motion_flag",
    "rule_stop_hint",
    "has_any_anomaly_in_video"
  ],
  "label_source": "AI City 2021 Track4 train-anomaly-results.txt",
  "feature_source": "OpenCV frame-difference motion features on real CCTV clips",
  "note": "y=1 if window time falls inside annotated anomaly interval"
}
STEP 6 OK
